# 19 - PixInsight as referee: contract 1

**Purpose.** Establish that PixInsight and `astropix` are looking at the same pixels and
speaking the same units, and publish the one comparison contract 1 is for: how our spread
estimator relates to PixInsight's two noise estimators, measured on the same planes of the same
frames. This is the first notebook in the repo that runs anything outside Python, so it also
records what the harness found about driving PixInsight headless - the core build, the flags, the
job round-trip - because a contract measured against an unnamed build is not a contract.

Four claims inherited from the retired attempts are checked here and not taken on trust: that
PixInsight normalises by 65535 and not by full scale, that `SplitCFA` hands back the two greens
transposed, that its variance divides by n-1 where numpy divides by n, and what its noise
estimators report. The harness rules they imply - no unbounded wait, no reading the exit code,
every script reports by writing a file - live in `astropix/pixinsight.py` and `pjsr/NOTES.md`,
beside the code that obeys them.

**What it is not for.** Not registration, not integration, not `eta_comb`: those are the
*engine* half of build step 5 and they get their own notebook, because a referee and an engine
are two purposes and a notebook with two purposes has none. Not a pair difference either - the
16-bit subtraction trap is real and unchecked here, because nothing in contract 1 subtracts.
Not contract 2 (`g` and `R` against PI's own estimate) and not contract 3 (stacking noise
reduction). Nothing here re-measures a published constant; `g`, `R` and the pedestal are inputs,
and only two of them are even read.

**Two frames, not a sweep.** A bias and a light, both from `data/` where the headers are trusted
in full because a protocol set them. The bias is where PixInsight's noise estimators and ours are
measuring the same thing - there is no structure to disagree about. The light is where they
stop, because stars and a sky gradient are signal to one estimator and outliers to another. One
frame would have shown half of it.

## 1. The harness answers

Before any number is compared, the thing producing half of them has to identify itself.

`pjsr/probe.js` measures nothing. It reports who answered, from where, and whether a real frame
opens. It is the first script in `pjsr/` and the one to re-run after any PixInsight upgrade,
because when a later script fails this is the known-good baseline it fails against.

Three claims are settled by this one cell. That the command line works at all - and every flag in
it earns its place, `-n` above all, without which PixInsight hands the run to whatever GUI session
happens to be open. That parameters passed as JSON keep their types, where PixInsight's own
command-line mechanism would deliver everything as a string. And that PixInsight opens a CFA
frame as a single-channel mosaic without debayering it, which is the precondition for every
comparison below: an interpolated pixel's noise is correlated with its neighbours', and comparing
against one would be comparing against a different sensor.

In [ ]:
import json
import pathlib
import sys

import numpy as np
import pandas as pd
from astropy.io import fits as _afits

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import fits as F, pixinsight as PI, spatial, stats

ROOT = pathlib.Path.cwd().parent
RESULTS = ROOT / "results"
DATA = ROOT / "data"

PLANES = list(spatial.PLANES)
FULL_SCALE = 4095                       # ADC counts; the units rule in CLAUDE.md
MAD_TO_SIGMA = stats.MAD_TO_SIGMA       # 1.4826, the same constant the index uses

# The two frames, named once.  Both from data/, so the headers are trusted in
# full -- including the type label, which is what a protocol set.
BIAS = DATA / "session02" / "frames" / "bias_g200_000.fits"
LIGHT = sorted((DATA / "session06").glob("*.fit"))[0]
FRAMES = {"bias": BIAS, "light": LIGHT}

CONTRACT_CSV = RESULTS / "pi_contract.csv"
CONSTANTS = RESULTS / "pi_contract_constants.json"

for name, path in FRAMES.items():
    assert path.is_file(), f"{name} frame missing: {path}"

probe = PI.run(PI.scripts_dir() / "probe.js", {"frame": PI.pi_path(LIGHT),
                                               "an_int": 42, "a_float": 1.5,
                                               "a_string": "42"})
assert probe["ok"], probe.get("error")

core = probe["core"]
print(f"PixInsight {core['version']} build {core['build']}, instance slot {core['instance']}")
print(f"working directory it woke up in: {core['working_directory']}")
print()
print("job round-trip -- the reason parameters travel as JSON and not on the command line:")
for k, v in probe["echo"].items():
    shown = v["value"] if k != "frame" else pathlib.Path(v["value"]).name
    print(f"  {k:10s} {str(shown)[:48]:50s} arrived as {v['type']}")
assert probe["echo"]["an_int"]["type"] == "number", "an integer arrived as something else"
assert probe["echo"]["a_string"]["type"] == "string"

### Did it debayer?

`channels` is the whole question. One channel means PixInsight opened the mosaic exactly as we
read it and the comparison is like-for-like. Three would mean it interpolated on load, and every
variance below would be measuring an interpolation kernel rather than a sensor.

In [ ]:
fr = probe["frame"]
print(f"{pathlib.Path(fr['path']).name}")
print(f"  {fr['width']} x {fr['height']}, {fr['channels']} channel(s), "
      f"{fr['bits_per_sample']} bits per sample, real={fr['is_real']}")

assert probe["frame_opened"], "PixInsight could not open the frame"
assert fr["channels"] == 1, ("PixInsight debayered on load; every comparison below is void, "
                             "because interpolated pixels have correlated noise (CLAUDE.md)")
mosaic, header = F.read(LIGHT)
assert (fr["width"], fr["height"]) == (mosaic.shape[1], mosaic.shape[0]), \
    "PixInsight and astropy disagree about the frame's geometry"
print(f"\nsame geometry as our reader, one channel, not debayered -- the comparison is "
      f"like-for-like")

## 2. The two frames, from their headers

Capture settings, trusted: a protocol set them. Nothing here reads the type label to decide
anything - the frames were chosen by name, from sessions whose protocols are in `protocols/`.

The bias is session 02's, gain 200 at the project's -10 C setpoint: the session that published
`g` and `R`, so a disagreement here is a disagreement about the numbers that matter most. The
light is session 06's first kept frame, 120 s on NGC 7000 - and shot at -20 C, which this
notebook does not care about because it converts no counts to electrons.

In [ ]:
rows = []
for name, path in FRAMES.items():
    h = _afits.getheader(path)
    rows.append({"frame": name, "file": path.name, "gain": int(h["GAIN"]),
                 "offset": int(h["OFFSET"]), "exptime": float(h["EXPTIME"]),
                 "set_temp": float(h["SET-TEMP"]), "ccd_temp": float(h["CCD-TEMP"]),
                 "date_obs": str(h["DATE-OBS"])[:19]})
chosen = pd.DataFrame(rows).set_index("frame")
print(chosen.to_string())

## 3. Contract 1a - the same pixels, in the same order

`pjsr/frame_stats.js` opens each frame, splits it with PixInsight's own `SplitCFA`, and reports
five statistics per plane plus the mosaic. It compares nothing: a referee that knew which answer
we wanted would not be one.

The claim being checked is that `SplitCFA` enumerates the 2x2 tile in PixInsight's `(x, y)` order
with `y` varying fastest, and that PI's first coordinate is the column - so in numpy's
`(row, col)` terms **the two greens come back transposed**. R and B sit on the diagonal and are
unmoved, which is what makes the error subtle enough to survive.

The claim also says how to check it, and the instruction is emphatic: **do not compare medians**.
Section 3.2 does it both ways and shows why.

In [ ]:
def describe_ours(plane):
    """Our numbers for one plane, in PixInsight's conventions and our units.

    `matching_stats` carries ddof=1 because pcl::Variance divides by n-1 where
    numpy divides by n -- section 5 measures what that is worth here.
    """
    return PI.matching_stats(stats.to_adc(plane))


pi_raw, ours = {}, {}
for name, path in FRAMES.items():
    r = PI.run(PI.scripts_dir() / "frame_stats.js", {"frame": PI.pi_path(path)})
    assert r["ok"], f"{name}: {r.get('error')}"
    assert r["core"]["build"] == core["build"], "a different core answered mid-notebook"
    pi_raw[name] = r
    mos, _ = F.read(path)
    ours[name] = {p: describe_ours(pl) for p, pl in spatial.split(mos).items()}
    ours[name]["mosaic"] = PI.matching_stats(stats.to_adc(mos))
    print(f"{name:6s} ok -- {r['mosaic']['n']:,} px mosaic, four planes of "
          f"{r['planes']['cfa0']['n']:,}")

### 3.1 The mapping, checked on the statistic that can see it

`min` is the cheapest statistic a single displaced pixel can move. If our plane and PixInsight's
plane are the same set of pixels, their minima are the same number exactly - not close, the same,
because both are one pixel's value and no arithmetic has touched it.

In [ ]:
STATS = ["min", "max", "mean", "median", "std"]

recs = []
for name in FRAMES:
    for i, plane in enumerate(PI.CFA_ORDER):
        key = f"cfa{i}"
        row = {"frame": name, "cfa": key, "plane": plane}
        for s in STATS:
            row[f"pi_{s}"] = float(PI.to_adc(pi_raw[name]["planes"][key][s]))
            row[f"ours_{s}"] = ours[name][plane][s]
        recs.append(row)
compare = pd.DataFrame(recs)
compare["d_min"] = compare.pi_min - compare.ours_min
compare["d_std_ppm"] = 1e6 * (compare.pi_std / compare.ours_std - 1.0)

print(compare[["frame", "cfa", "plane", "pi_min", "ours_min", "d_min"]].to_string(index=False))
worst = compare.d_min.abs().max()
print(f"\nlargest minimum disagreement: {worst:.9f} ADC counts")
assert worst < 1e-6, ("our planes are not PixInsight's planes -- check the CFA order before "
                      "believing anything below")

### 3.2 What the naive mapping looks like, and why the median hides it

Under a naive reading of RGGB the two greens swap. This cell scores both mappings on the minimum
and on the median, so the claim's warning is a measurement rather than a caution.

In [ ]:
NAIVE = ("R", "G1", "G2", "B")

for name in FRAMES:
    print(f"-- {name}")
    for label, order in (("PI order (R, G2, G1, B)", PI.CFA_ORDER), ("naive  (R, G1, G2, B)", NAIVE)):
        bad_min = bad_med = 0
        for i, plane in enumerate(order):
            pl = pi_raw[name]["planes"][f"cfa{i}"]
            bad_min += abs(float(PI.to_adc(pl["min"])) - ours[name][plane]["min"]) > 1e-6
            bad_med += abs(float(PI.to_adc(pl["median"])) - ours[name][plane]["median"]) > 1e-6
        print(f"   {label:24s} planes wrong by minimum: {bad_min}   by median: {bad_med}")
print()
print("The greens' own numbers, where the difference lives:")
g = compare[(compare.plane.isin(["G1", "G2"]))]
print(g[["frame", "cfa", "plane", "pi_min", "pi_median"]].to_string(index=False))

## 4. Contract 1b - the unit boundary

PixInsight normalises pixel data to [0, 1]. The divisor is the **container** maximum 65535 - not
full scale, and not the sensor's saturation level. This project's unit is the stored value over
16, so a PixInsight value converts as `v * 65535 / 16`, and the composed factor is **4095.9375**.

Multiplying by 4095 instead is wrong by 0.023%. That is the reason this needs a measurement and
not a comment: the error is far too small to look like an error.

The check is the cheapest one available. Take the median PixInsight reports, multiply by 65535,
and see whether an integer comes back - and whether it is the integer our reader independently
reads off the same file.

In [ ]:
unit_rows = []
for name, path in FRAMES.items():
    mos, _ = F.read(path)
    pi_med = pi_raw[name]["mosaic"]["median"]
    unit_rows.append({
        "frame": name,
        "pi_median_normalised": pi_med,
        "x65535": pi_med * 65535,
        "our_stored_median": float(np.median(mos)),
        "pi_to_adc": float(PI.to_adc(pi_med)),
        "our_adc_median": float(np.median(stats.to_adc(mos))),
        "x4095_the_wrong_one": pi_med * 4095,
    })
units = pd.DataFrame(unit_rows).set_index("frame")
print(units.T.to_string())

for name in FRAMES:
    r = units.loc[name]
    assert abs(r.x65535 - round(r.x65535)) < 1e-4, "PI's value is not a stored integer over 65535"
    assert abs(r.pi_to_adc - r.our_adc_median) < 1e-6, "the two tools disagree about the median"
err = float((units.x4095_the_wrong_one / units.pi_to_adc - 1).abs().max()) * 100
print(f"\n65535/16 = {65535/16}.  Using 4095 instead is off by {err:.3f}% -- "
      f"small enough to pass for rounding, which is the danger")

## 5. Contract 1c - whose variance?

`pcl::Variance` returns the sample variance, dividing by n-1. `ndarray.std()` returns the
population one, dividing by n. The ratio is `sqrt(n/(n-1))`.

On a sub-plane of 2,073,600 pixels that is 1 + 2.4e-7, so this is not a correction anyone would
notice on a real frame - and that is exactly the point. It is invisible at full-frame n and 4%
at the dozen values a test uses, so it has to be established somewhere it can be seen and then
carried everywhere. `pixinsight.matching_stats` carries it; the test at small n proves it.

In [ ]:
n = pi_raw["bias"]["planes"]["cfa0"]["n"]
print(f"n per plane = {n:,};  sqrt(n/(n-1)) - 1 = {np.sqrt(n/(n-1)) - 1:.3e}")
print(f"the ddof=1 correction is worth {1e6*(np.sqrt(n/(n-1)) - 1):.3f} ppm on a plane this size")
print()
print("our std vs PI's, in parts per million:")
print(compare[["frame", "cfa", "plane", "pi_std", "ours_std", "d_std_ppm"]].to_string(index=False))

worst_ppm = compare.d_std_ppm.abs().max()
print(f"\nlargest disagreement: {worst_ppm:.3f} ppm")
assert worst_ppm < 10.0, "the standard deviations disagree by more than float noise"

## 6. Contract 1d - three estimators, one plane

This is what contract 1 exists to produce. Everything above establishes that the two tools are
looking at the same pixels in the same units; this section asks what they *say* about them.

Four numbers per plane:

- **PI MRS** - multiresolution support. `frame_stats.js` chooses the layer count the way
  PixInsight's own `NoiseEvaluation.js` does: decreasing layers, accepting the first whose noisy
  pixel set reaches 1% of the frame, falling back to k-sigma if none does.
- **PI k-sigma** - iterative clipping about the mean.
- **ours** - `1.4826 x MAD`, which is the estimator the frame index and every bench session use.
- **PI's plain standard deviation**, which rejects nothing and is here as the ceiling.

The prediction inherited from the retired project was an ordering: PI's multiresolution rejects
harder than our clip, PI's plain sigma rejects nothing, and ours should sit between them. It is
a prediction with no surviving frames behind it, so what follows is the measurement.

In [ ]:
noise_rows = []
for name in FRAMES:
    mos, _ = F.read(FRAMES[name])
    planes = spatial.split(mos)
    for i, plane in enumerate(PI.CFA_ORDER):
        key = f"cfa{i}"
        pl = stats.to_adc(planes[plane]).astype(np.float64)
        med = float(np.median(pl))
        our_mad = float(np.median(np.abs(pl - med)))
        nz = pi_raw[name]["noise"]["planes"][key]
        noise_rows.append({
            "frame": name, "cfa": key, "plane": plane,
            "pi_mrs": float(PI.to_adc(nz["mrs"])),
            "pi_mrs_layers": nz["mrs_layers"],
            "pi_ksigma": float(PI.to_adc(nz["ksigma"])),
            "pi_chosen": nz["chosen"],
            "ours_mad_sigma": MAD_TO_SIGMA * our_mad,
            "our_mad": our_mad,
            "pi_mad": float(PI.to_adc(pi_raw[name]["planes"][key]["mad"])),
            "pi_std": float(PI.to_adc(pi_raw[name]["planes"][key]["std"])),
        })
noise = pd.DataFrame(noise_rows)
noise["d_mad"] = noise.pi_mad - noise.our_mad

cols = ["frame", "plane", "pi_mrs", "pi_ksigma", "ours_mad_sigma", "pi_std", "our_mad", "pi_mad"]
print(noise[cols].to_string(index=False, float_format=lambda v: f"{v:9.4f}"))

### 6.1 The arithmetic agrees; the estimators do not

Two different things can make our sigma differ from PixInsight's. One is that we computed the
same quantity differently, which would be a bug. The other is that we computed a different
quantity, which is a finding.

`MAD` separates them. PixInsight reports its own median absolute deviation, and so do we, so
these are the same definition on the same pixels. If they agree, every remaining difference is
about *which estimator*, not about arithmetic.

In [ ]:
print(noise[["frame", "plane", "our_mad", "pi_mad", "d_mad"]].to_string(index=False))
worst_mad = noise.d_mad.abs().max()
print(f"\nlargest MAD disagreement: {worst_mad:.9f} ADC counts")
assert worst_mad < 1e-6, ("our MAD is not PixInsight's MAD -- the disagreement below would be "
                          "arithmetic and not estimator choice")
print("\nSo the spread numbers below differ because the estimators differ, and for no other "
      "reason.")

### 6.2 The ordering, measured

The inherited prediction was `MRS < ours < k-sigma`, with the spread widening as gain amplifies
the outlier tail. Whether that holds is a question about the frame, not about the sensor: an
estimator that rejects structure and an estimator that does not can only disagree where there is
structure to reject.

In [ ]:
def ordering(row):
    named = {"MRS": row.pi_mrs, "ksigma": row.pi_ksigma, "ours": row.ours_mad_sigma}
    return " < ".join(k for k, _ in sorted(named.items(), key=lambda kv: kv[1]))


noise["ordering"] = noise.apply(ordering, axis=1)
noise["ours_over_mrs"] = noise.ours_mad_sigma / noise.pi_mrs

print(noise[["frame", "plane", "pi_mrs", "pi_ksigma", "ours_mad_sigma", "ordering",
             "ours_over_mrs"]].to_string(index=False, float_format=lambda v: f"{v:8.4f}"))
print()
for name in FRAMES:
    sub = noise[noise.frame == name]
    print(f"{name:6s}: {sorted(set(sub.ordering))}, "
          f"ours/MRS {sub.ours_over_mrs.min():.3f} to {sub.ours_over_mrs.max():.3f}")
print(f"\nPI chose: {sorted(set(noise.pi_chosen))}, at "
      f"{sorted(set(noise.pi_mrs_layers))} layers -- the k-sigma fallback was "
      f"{'taken' if 'ksigma' in set(noise.pi_chosen) else 'never taken'}")

### 6.3 The floor under our estimator

A median absolute deviation over integer data is itself an integer or a half-integer. Our sigma is
`1.4826 x MAD`, so it can only land on multiples of 1.4826 ADC counts. Eight planes above produced
three distinct MADs between them: 1, 2 and 3.

The grid is **absolute**, so what matters is how it compares with the spread being measured, and
the answer depends entirely on where in the gain range you stand. `results/bias_sweep.csv` has
read noise at 77 gains, measured two independent ways - `R_sd` from pair differences, which is not
quantised, and `R_mad`, which is. The second cell below reads both back and asks the sharp
question: **over what range of true read noise does `R_mad` return the same answer?**

The grid there is 1.0483 rather than 1.4826, and the factor is `sqrt(2)`: `R_mad` is a
*pair-difference* MAD divided by `sqrt(2)` to recover a single frame's noise, so the step comes
down with it. Same phenomenon, one frame's worth smaller.

In [ ]:
step = MAD_TO_SIGMA
noise["mad_grid_units"] = noise.ours_mad_sigma / step
print(noise[["frame", "plane", "our_mad", "ours_mad_sigma", "mad_grid_units"]].to_string(
    index=False, float_format=lambda v: f"{v:9.4f}"))
print(f"\nour sigma can only land on multiples of {step} ADC counts;")
print(f"distinct MAD values across all eight planes: {sorted(set(noise.our_mad))}")
print(f"PI's MRS, which is not quantised this way: "
      f"{noise.pi_mrs.min():.4f} to {noise.pi_mrs.max():.4f}")

In [ ]:
# Read back, not restated: the prose above claims a grid and this is the cell
# that can contradict it.  One row per gain at the lowest offset present.
sweep = pd.read_csv(RESULTS / "bias_sweep.csv")
per_gain = (sweep.sort_values("offset").groupby("gain").first().reset_index()
                 .dropna(subset=["R_mad", "R_sd"]))
per_gain = per_gain[per_gain.gain <= 450]           # the gain domain, CLAUDE.md
per_gain["rung"] = per_gain.R_mad.round(4)

by_rung = per_gain.groupby("rung").agg(gains=("gain", "size"),
                                       gain_lo=("gain", "min"), gain_hi=("gain", "max"),
                                       sd_lo=("R_sd", "min"), sd_hi=("R_sd", "max"))
by_rung["sd_spread"] = by_rung.sd_hi / by_rung.sd_lo
print(f"{len(per_gain)} gains at or below 450 in results/bias_sweep.csv\n")
print("each value R_mad can take, and the true read noise it covers:")
print(by_rung.head(8).to_string(float_format=lambda v: f"{v:8.3f}"))

worst = by_rung.sd_spread.idxmax()
w = by_rung.loc[worst]
print(f"\nworst rung: R_mad = {worst} is returned for {int(w.gains)} gains "
      f"({int(w.gain_lo)} to {int(w.gain_hi)}),")
print(f"  over which the pair-difference read noise runs {w.sd_lo:.3f} to {w.sd_hi:.3f} "
      f"ADC counts -- a factor of {w.sd_spread:.2f}.")
print(f"  R_mad calls those {int(w.gains)} gains identical; R_sd resolves every one of them.")
print()
shown = per_gain[per_gain.gain.isin([0, 50, 100, 190, 200, 250, 300, 450])]
print(shown[["gain", "offset", "R_sd", "R_mad"]].to_string(index=False,
                                                           float_format=lambda v: f"{v:8.3f}"))
print(f"\nthe drop at gain 200 is the HCG branch, not an error: "
      f"{float(per_gain[per_gain.gain == 190].R_sd.iloc[0]):.2f} at gain 190 to "
      f"{float(per_gain[per_gain.gain == 200].R_sd.iloc[0]):.2f} at gain 200 -- and note that "
      f"R_mad\nreturns the same value at gain 0 and at gain 250, where the truth differs by "
      f"{float(per_gain[per_gain.gain == 250].R_sd.iloc[0]) / float(per_gain[per_gain.gain == 0].R_sd.iloc[0]):.1f}x.")

### 6.4 The inherited prediction is refuted, and the mechanism is not rejection

The claim was `MRS < ours < ksigma`. On both frames and on all eight planes, **ours is above
both**, by a factor of 1.21 to 1.49. The two PixInsight estimators land within 10% of each other
and ours sits half again as high.

It is not a rejection difference, because section 6.1 already showed our MAD and PixInsight's MAD
are the same number to seven decimal places. It is the grid. On the bias, `MAD` is pinned at
exactly 1 count on every plane, so `1.4826 x MAD` is pinned at 1.4826 - while PixInsight's own
plain standard deviation of the same pixels is about 1.09. Our estimator is roughly a third high
there, and it could not have been anything else: the nearest values it can take are 0 and 2.965.

**What this does not touch.** No published constant in this repo comes from a spatial MAD. Read
noise is published from pair differences (`bias_sweep.csv`, `ptc_constants.json`) and from the
temporal spread across a bias block (`cold_constants.json`), both of which sidestep the grid
entirely. What it does touch is the `sigma` column of the frame index, which is a *description* of
a frame and not evidence about the sensor - and which is now known to read high wherever the
spread is a few counts.

## 7. What is published

Two files. `pi_contract.csv` is the per-plane record behind everything above - both tools' five
statistics, both tools' MAD, and the three noise estimators, in ADC counts. `pi_contract_constants.json`
carries the four contract results with provenance.

The core build is part of the provenance and not a footnote. Every number here is a statement
about PixInsight 1.9.2 build 1632 driven a particular way, and a different build is a different
referee.

In [ ]:
table = compare.merge(noise.drop(columns=["cfa", "pi_chosen", "ordering", "pi_std"]),
                      on=["frame", "plane"], how="outer")
table.insert(0, "pi_build", core["build"])
table.insert(0, "pi_version", core["version"])
table.to_csv(CONTRACT_CSV, index=False)
print(f"wrote {CONTRACT_CSV.name}: {len(table)} rows, {len(table.columns)} columns")
print(table.columns.tolist())

In [ ]:
measured_on = "2026-09-19"
n_frames = len(FRAMES)


def constant(value, unit, uncertainty, note):
    return {"value": value, "unit": unit, "uncertainty": uncertainty,
            "source_frames": n_frames, "measured_on": measured_on,
            "notebook": "19_pi_contract.ipynb", "note": note}


def nn(v):
    return None if v is None or not np.isfinite(v) else round(float(v), 9)


by_frame = {name: {r.plane: {"mrs": nn(r.pi_mrs), "ksigma": nn(r.pi_ksigma),
                             "ours": nn(r.ours_mad_sigma), "pi_std": nn(r.pi_std)}
                   for r in noise[noise.frame == name].itertuples()}
            for name in FRAMES}

constants = {
    "referee": constant(
        {"application": "PixInsight", "version": core["version"], "build": core["build"],
         "command": "PixInsight.exe -n --automation-mode --run=<abs path> --force-exit"},
        None, None,
        "the build every number in this file was measured against, and the command line that "
        "reached it.  A different build is a different referee; re-run pjsr/probe.js after any "
        "upgrade.  The flags are not decoration -- without -n PixInsight yields to an already "
        "running instance and the run is handed to whatever GUI session is open.  See "
        "pjsr/NOTES.md"),
    "unit_scale": constant(
        65535 / 16, "ADC counts per unit PixInsight value", 0.0,
        "PixInsight normalises by the CONTAINER maximum 65535, not by full scale 4095 and not by "
        "the sensor's saturation level 65520.  This project's unit is stored/16, so the composed "
        "factor is 65535/16 = 4095.9375.  Verified on both frames: PI's reported median times "
        "65535 is our reader's stored median, exactly.  Using 4095 is wrong by 0.023%, which is "
        "small enough to pass for rounding -- which is why it is published as a constant rather "
        "than left as a remark"),
    "cfa_order": constant(
        list(PI.CFA_ORDER), None, None,
        "the plane each SplitCFA output holds, in output order CFA0..CFA3.  PI enumerates the "
        "2x2 tile in (x, y) with y fastest and its first coordinate is the column, so in numpy "
        "(row, col) terms THE TWO GREENS ARE TRANSPOSED against a naive RGGB reading; R and B "
        "sit on the diagonal and are unmoved.  Verified on the minimum, which is one pixel and "
        "cannot hide a displacement.  The medians agree under BOTH mappings on both frames, so a "
        "median comparison would have passed the wrong mapping"),
    "split_agreement": constant(
        {"max_abs_min_diff": nn(compare.d_min.abs().max()),
         "max_abs_mad_diff": nn(noise.d_mad.abs().max()),
         "max_abs_std_diff_ppm": nn(compare.d_std_ppm.abs().max())},
        "ADC counts; ppm for the last", None,
        "contract 1's precondition: astropix.spatial.split and PixInsight's SplitCFA return the "
        "same pixels.  Minima and MADs agree exactly; the standard deviations agree to float "
        "noise once ddof=1 is carried on our side, because pcl::Variance divides by n-1 where "
        "numpy divides by n.  At 2,073,600 px per plane that correction is 0.24 ppm and could "
        "not have been established here -- it is proved at small n in tests/test_pixinsight.py "
        "and carried in pixinsight.matching_stats"),
    "noise_estimators": constant(
        by_frame, "ADC counts", None,
        "contract 1's result: PI's multiresolution-support and k-sigma noise estimates, our "
        "1.4826 x MAD, and PI's plain standard deviation, per CFA plane, on one bias (session 02, "
        "gain 200, -10 C) and one light (session 06, 120 s, -20 C).  Measured on the frames in "
        "hand and not extrapolated: two frames say what two frames can say"),
    "noise_ordering": constant(
        {"measured": {name: sorted(set(noise[noise.frame == name].ordering)) for name in FRAMES},
         "ours_over_mrs": {"min": nn(noise.ours_over_mrs.min()),
                           "max": nn(noise.ours_over_mrs.max())},
         "predicted": "MRS < ours < ksigma",
         "verdict": "refuted"},
        None, None,
        "the inherited prediction was MRS < ours < ksigma, with ours sitting between PI's two "
        "estimators.  REFUTED on both frames and all eight planes: ours is above BOTH, by 1.21 "
        "to 1.49x, and PI's two land within 10% of each other.  The mechanism is not rejection "
        "-- our MAD and PI's MAD agree to 2e-7 counts -- it is the quantisation grid below.  The "
        "estimators can only disagree where there is structure to reject, so the ORDER of PI's "
        "two is a statement about these two frames; the position of ours above them is not"),
    "our_sigma_quantisation": constant(
        nn(MAD_TO_SIGMA), "ADC counts",  None,
        "the step our estimator can land on.  A MAD over integer data is an integer or a half "
        "integer, so 1.4826 x MAD is quantised -- and read noise on this sensor is about one "
        "count at gain 0 and one at gain 200, so the grid is coarse where it matters most.  Not "
        "a disagreement with PixInsight: PI's MAD and ours agree to 2e-7 counts.  Independently "
        "visible in results/bias_sweep.csv, where R_mad takes a handful of values across 77 "
        "gains while R_sd moves smoothly -- there on a grid of 1.0483, smaller by sqrt(2) "
        "because it is a pair-difference MAD.  NO PUBLISHED CONSTANT IN THIS REPO IS AFFECTED: "
        "read noise is published from pair differences and from the temporal spread across a "
        "bias block, both of which sidestep the grid.  What is affected is the frame index's "
        "`sigma` column, which describes a frame and is not evidence about the sensor, and which "
        "reads high wherever the spread is a few counts.  PI's MRS is not quantised this way, "
        "which is what makes it useful as a referee"),
}

with open(CONSTANTS, "w", encoding="utf-8") as fh:
    json.dump(constants, fh, indent=2)
print(f"wrote {CONSTANTS.name}: {len(constants)} constants")
for k, v in constants.items():
    print(f"  {k:26s} {str(v['value'])[:70]}")

## 8. What contract 1 does and does not license

**Licensed.** PixInsight and `astropix` read the same pixels, split them into the same planes,
and convert between their units without loss. Any number either tool produces about this sensor's
frames can now be compared with the other's, and `pjsr/` has a harness that fails in bounded time
and says why.

**Refuted.** The inherited ordering of the noise estimators. Ours does not sit between
PixInsight's two; it sits above both, on every plane of both frames, and the cause is our
estimator's quantisation grid rather than any difference in what gets rejected. That is worth
having as a measurement: it says the index's `sigma` column reads high at small spreads, and it
says PixInsight's MRS is the better referee for a spread of a few counts precisely because it is
not built on a median of integers.

**Not licensed.** Nothing here was subtracted, stacked, registered or integrated. Contract 2
(`g` and `R` against PixInsight's own estimate) needs a bias pair, and a pair difference taken in
16-bit clips every negative value and **halves the apparent read noise while looking entirely
plausible** - the one inherited claim this notebook leaves unchecked, because contract 1 has
nothing to subtract. Contract 3 needs `ImageIntegration`. `eta_comb`, the SNR estimator's
repeatability and MISSION's four ranked pairs all still wait on the engine half of build step 5.

**Two frames are two frames.** The noise-estimator comparison is measured, not extrapolated. It
says what these estimators do on a bias at gain 200 and on a 120 s light, and a claim about any
other gain or exposure would be a claim this notebook did not test.